In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
keep_cols = ["age", "candidate", "ratio_adj_p28"]

df = pd.read_excel("wb-dev-data/raw_data_candidate_dev_analysis.xlsx")
df = df.loc[:, keep_cols]
df.head(10)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

age_order = ["P1", "P7", "P14", "P21", "P28"]

for cand in df["candidate"].unique():
    d = df[df["candidate"] == cand].copy()
    d["age"] = pd.Categorical(d["age"], categories=age_order, ordered=True)

    fig, ax = plt.subplots(figsize=(4.8, 3.6))

    # Grey CI band + grey line (mean)
    sns.lineplot(
        data=d, x="age", y="ratio_adj_p28",
        estimator="mean",
        errorbar=("ci", 95),   
        n_boot=2000,
        color="0.6",          
        linewidth = 2.0,
        ax=ax
    )

    # Make CI band grey explicitly
    for coll in ax.collections:
        coll.set_alpha(0.17)   # light CI
        coll.set_facecolor("0.7")
        coll.set_edgecolor("none")

    # raw points in black
    sns.stripplot(
        data=d, x="age", y="ratio_adj_p28",
        order=age_order,
        jitter=0.1, alpha=1,
        color="black",
        ax=ax,
        zorder = 4
    )

    # mean markers (black)
    means = d.groupby("age", observed=True)["ratio_adj_p28"].mean().reindex(age_order)
    x = np.arange(len(age_order))
    ax.plot(
        x, means.values,
        marker="o", linestyle="None",
        markersize=10,
        markerfacecolor="grey",
        markeredgecolor="grey",
        markeredgewidth=None,
        zorder=3
    )

    ax.set_title(cand)
    ax.set_xlabel("Postnatal age")
    ax.set_ylabel("Normalized WB (total protein + gel calibrator)")
    plt.tight_layout()

    fig.savefig(
    f"{cand}_development_WB.svg",
    format="svg",
    bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)
